# Part 4: Graph Modularity and Community Detection

## Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
from scipy.cluster.hierarchy import linkage, dendrogram
from scipy.spatial.distance import squareform

**(a)** Define in your own words the notion of modularity of a network and how it can be used to split a network into communities. Should your description include whether the modularity depends solely on the structure of the network? **(0.5 points)**


**(b)** The adjacency matrix of a 6-node directed graph is given by:
$$
\mathbf{A} = \left[
\begin{array}{cccccc}
0 & 1 & 1 & 0 & 0 & 0 \\
1 & 0 & 1 & 0 & 0 & 0 \\
1 & 1 & 0 & 1 & 0 & 0 \\
0 & 0 & 1 & 0 & 1 & 1 \\
0 & 0 & 0 & 1 & 0 & 1 \\
0 & 0 & 0 & 1 & 1 & 0
\end{array}
\right]
$$
Build the graph, compute the modularity matrix $\mathbf{B}$, its leading eigenvector, and use its signs to choose the membership of each node. Calculate the modularity $Q$ of the resulting split. **(0.5 points)**

## Question 1

### Question (b): Build the 6-node adjacency matrix

In [ ]:
A = np.zeros((6, 6), dtype=float)

# node 0 (R node 1)
A[0, 1] = 1
A[0, 2] = 1
# node 1 (R node 2)
A[1, 0] = 1
A[1, 2] = 1
# node 2 (R node 3)
A[2, 0] = 1
A[2, 1] = 1
A[2, 3] = 1
# node 3 (R node 4)
A[3, 2] = 1
A[3, 4] = 1
A[3, 5] = 1
# node 4 (R node 5)
A[4, 3] = 1
A[4, 5] = 1
# node 5 (R node 6)
A[5, 3] = 1
A[5, 4] = 1

print("Adjacency matrix A:")
print(A)

### Question (b): Build and plot the graph

In [ ]:
G = nx.from_numpy_array(A, create_using=nx.DiGraph)

fig, ax = plt.subplots(figsize=(5, 4))
pos = nx.spring_layout(G, seed=42)
nx.draw(G, pos, ax=ax, with_labels=True, node_color='lightblue',
        node_size=600, arrows=True, arrowsize=15)
ax.set_title('6-node directed graph')
plt.tight_layout()
plt.show()

### Question (b): Compute the modularity matrix B

In [ ]:
D_deg = A.sum(axis=1)          # degree vector
m     = A.sum() / 2            # total number of (undirected) edges
B     = A - np.outer(D_deg, D_deg) / (2 * m)

print("Modularity matrix B:")
print(np.round(B, 4))

### Question (b): Leading eigenvector and community membership

In [ ]:
w, v = np.linalg.eig(B)

# Sort eigenvalues descending and take the leading eigenvector
sort_idx       = np.argsort(w)[::-1]
w_sorted       = w[sort_idx].real
v_sorted       = v[:, sort_idx].real
leading_vector = v_sorted[:, 0]

print("Eigenvalues (descending):", np.round(w_sorted, 4))
print("Leading eigenvector:      ", np.round(leading_vector, 4))

In [ ]:
# Assign membership by sign of leading eigenvector
# Negative -> community 1, Positive -> community 2  (matching R convention)
membership = np.where(leading_vector < 0, 1, 2)
print("Community membership (1-indexed nodes):")
for node, comm in enumerate(membership):
    print(f"  Node {node + 1}: community {comm}")

### Question (b): Compute modularity Q manually

In [ ]:
# Q = (1 / 2m) * sum_{ij} B_ij * delta(c_i, c_j)
Q = 0.0
for i in range(6):
    for j in range(6):
        if membership[i] == membership[j]:
            Q += B[i, j]
Q /= (2 * m)
print(f"Modularity Q = {Q:.4f}")

**(c)** Interpret the **magnitude of the coordinates** of the leading eigenvector. Which nodes have smaller magnitudes and why? **(0.5 points)**

**(d)** If we wanted to split the network into more than two classes, how could we extend the spectral algorithm? **(0.25 points)**

**(e)** What would happen if all the eigenvalues of the modularity matrix were smaller than zero? What would this indicate in terms of the structure of the network? **(0.25 points)**

---

## Question 2 (2 points)

**(a)** Build the adjacency matrix for the following 5-node graph:
- Node 1: edges to 2, 4
- Node 2: edges to 1, 3, 4
- Node 3: edges to 2, 4, 5
- Node 4: edges to 1, 2, 3, 5
- Node 5: edges to 3, 4

Create a similarity matrix $\boldsymbol{\Sigma}$ where each coordinate $(i,j)$ is the cosine similarity between the adjacency vectors of nodes $i$ and $j$. **(1 point)**

**(b)** Using the distance $\mathbf{D} = 1 - \boldsymbol{\Sigma}$, perform hierarchical clustering with single linkage and plot the resulting dendrogram. **(1 point)**